In [1]:
# IMPORT LIBRARIES

In [2]:
import os
import glob
import numpy as np

import music21

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

import warnings
warnings.filterwarnings("ignore")

In [3]:
# LOAD MIDI FILES

In [4]:
midi_files = glob.glob("midi_songs/*.mid")

print("Total MIDI Files:", len(midi_files))

Total MIDI Files: 30


In [5]:
# EXTRACT NOTES FROM MIDI FILES

In [6]:
notes = []

for file in midi_files:
    print("Reading:", file)
    try:
        midi = music21.converter.parse(file)

        for element in midi.flatten().notes:
            if isinstance(element, music21.note.Note):
                notes.append(str(element.pitch))

            elif isinstance(element, music21.chord.Chord):
                notes.append(".".join(str(n) for n in element.normalOrder))

    except Exception as e:
        print("Error in", file)
        print(e)

print("Total Notes:", len(notes))

Reading: midi_songs\ajeebdaastan.mid
Reading: midi_songs\ayemerehumsafar.mid
Reading: midi_songs\barbardekho.mid
Reading: midi_songs\bekararkarkehamein.mid
Reading: midi_songs\chhodayehum.mid
Reading: midi_songs\dilkyakare.mid
Reading: midi_songs\dummarodum.mid
Reading: midi_songs\ekladkikodekha.mid
Reading: midi_songs\hoothonsechoolo.mid
Reading: midi_songs\jabkoibaatbigadjaaye.mid
Reading: midi_songs\janaganamana.mid
Reading: midi_songs\kahidurjab.mid
Reading: midi_songs\kehnahikya.mid
Reading: midi_songs\kisikimuskarahaton.mid
Reading: midi_songs\kuchnaakahon.mid
Reading: midi_songs\lakhonhainyahan.mid
Reading: midi_songs\maineterliye.mid
Reading: midi_songs\mainkoiaisageet.mid
Reading: midi_songs\meinzindagikasaath.mid
Reading: midi_songs\merajootahain.mid
Reading: midi_songs\meranaamchinchinchu.mid
Reading: midi_songs\meresapnonkirani.mid
Reading: midi_songs\omjaijagdisha.mid
Reading: midi_songs\pukartachalahoonmain.mid
Reading: midi_songs\raheinnaraheinhum.mid
Reading: midi_songs

In [7]:
# CREATE UNIQUE NOTES

In [8]:
pitchnames = sorted(set(notes))

print("Unique Notes:", len(pitchnames))

Unique Notes: 274


In [9]:
# CREATE NOTE TO INTEGER MAPPING

In [10]:
note_to_int = dict((note, number) for number, note in enumerate(pitchnames))

print(note_to_int)

{'0': 0, '0.1': 1, '0.1.5': 2, '0.1.5.8': 3, '0.2': 4, '0.2.4.5': 5, '0.2.5': 6, '0.2.6': 7, '0.2.7': 8, '0.3': 9, '0.3.5': 10, '0.3.6.8': 11, '0.3.7': 12, '0.4': 13, '0.4.6': 14, '0.4.7': 15, '0.4.8': 16, '0.5': 17, '0.6': 18, '1': 19, '1.2': 20, '1.2.6.9': 21, '1.3': 22, '1.3.6': 23, '1.3.8': 24, '1.4': 25, '1.4.5.7': 26, '1.4.6': 27, '1.4.6.8': 28, '1.4.7': 29, '1.4.7.10': 30, '1.4.7.9': 31, '1.4.8': 32, '1.5': 33, '1.5.6': 34, '1.5.8': 35, '1.5.9': 36, '1.6': 37, '1.7': 38, '10': 39, '10.0': 40, '10.0.4': 41, '10.0.5': 42, '10.1': 43, '10.1.3.6': 44, '10.1.4.6': 45, '10.1.5': 46, '10.11': 47, '10.2': 48, '10.2.4': 49, '10.2.5': 50, '10.3': 51, '11': 52, '11.0': 53, '11.0.3.7': 54, '11.0.4.7': 55, '11.1': 56, '11.1.4': 57, '11.1.4.7': 58, '11.1.6': 59, '11.2': 60, '11.2.4': 61, '11.2.4.7': 62, '11.2.5': 63, '11.2.5.7': 64, '11.2.6': 65, '11.3': 66, '11.3.4': 67, '11.3.6': 68, '11.4': 69, '2': 70, '2.3': 71, '2.3.5.7': 72, '2.3.5.9.10': 73, '2.4': 74, '2.5': 75, '2.5.7': 76, '2.5.7.1

In [11]:
# CREATE INPUT AND OUTPUT SEQUENCES

In [12]:
sequence_length = 100

network_input = []
network_output = []

for i in range(len(notes) - sequence_length):
    sequence_in = notes[i:i + sequence_length]
    sequence_out = notes[i + sequence_length]

    network_input.append([note_to_int[n] for n in sequence_in])
    network_output.append(note_to_int[sequence_out])

print("Input Sequences:", len(network_input))
print("Output Sequences:", len(network_output))

Input Sequences: 39860
Output Sequences: 39860


In [13]:
# DISPLAY INPUT SHAPE

In [14]:
print(np.array(network_input).shape)

(39860, 100)


In [15]:
# NORMALIZE INPUT DATA

In [16]:
network_input = np.reshape(
    network_input,
    (len(network_input), sequence_length, 1)
)

network_input = network_input / float(len(pitchnames))

print(network_input.shape)

(39860, 100, 1)


In [17]:
# ONE HOT ENCODE OUTPUT

In [18]:
network_output = to_categorical(network_output)

print(network_output.shape)

(39860, 274)


In [19]:
# BUILD LSTM MODEL

In [20]:
model = Sequential()

model.add(LSTM(
    512,
    input_shape=(network_input.shape[1], network_input.shape[2]),
    return_sequences=True
))

model.add(Dropout(0.3))

model.add(LSTM(512))

model.add(Dropout(0.3))

model.add(Dense(256, activation="relu"))

model.add(Dense(network_output.shape[1], activation="softmax"))

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam"
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 100, 512)            │       1,052,672 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 100, 512)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 512)                 │       2,099,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 274)                 │          70,418 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,353,618 (12.79 MB)

 Trainable params: 3,353,618 (12.79 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
# TRAIN THE MODEL

In [22]:
history = model.fit(
    network_input,
    network_output,
    epochs=50,
    batch_size=64
)++


Epoch 1/50
623/623 ━━━━━━━━━━━━━━━━━━━━ 1439s 2s/step - loss: 4.4666
Epoch 2/50
623/623 ━━━━━━━━━━━━━━━━━━━━ 1555s 2s/step - loss: 4.4292
Epoch 3/50
623/623 ━━━━━━━━━━━━━━━━━━━━ 1041s 2s/step - loss: 4.4264
Epoch 4/50
623/623 ━━━━━━━━━━━━━━━━━━━━ 1065s 2s/step - loss: 4.4188
Epoch 5/50
623/623 ━━━━━━━━━━━━━━━━━━━━ 951s 2s/step - loss: 4.4072 
Epoch 6/50
623/623 ━━━━━━━━━━━━━━━━━━━━ 1002s 2s/step - loss: 4.2925
Epoch 7/50
623/623 ━━━━━━━━━━━━━━━━━━━━ 1018s 2s/step - loss: 4.1918
Epoch 8/50
623/623 ━━━━━━━━━━━━━━━━━━━━ 1103s 2s/step - loss: 4.1521
Epoch 9/50
623/623 ━━━━━━━━━━━━━━━━━━━━ 1006s 2s/step - loss: 4.0945
Epoch 10/50
623/623 ━━━━━━━━━━━━━━━━━━━━ 1025s 2s/step - loss: 4.0498
Epoch 11/50
623/623 ━━━━━━━━━━━━━━━━━━━━ 1189s 2s/step - loss: 3.9899
Epoch 12/50
623/623 ━━━━━━━━━━━━━━━━━━━━ 1233s 2s/step - loss: 3.9310
Epoch 13/50
623/623 ━━━━━━━━━━━━━━━━━━━━ 1015s 2s/step - loss: 3.8041
Epoch 14/50
623/623 ━━━━━━━━━━━━━━━━━━━━ 1012s 2s/step - loss: 3.5774
Epoch 15/50
623/623 ━━━━━━━━━

In [23]:
# SAVE TRAINED MODEL

In [24]:
model.save("music_generation_model.h5")

print("Model Saved Successfully!")

Model Saved Successfully!


In [25]:
# GENERATE MUSIC

In [26]:
start = np.random.randint(0, len(network_input)-1)

pattern = network_input[start]

prediction_output = []

In [27]:
# GENERATE 500 NOTES

In [28]:
# CREATE INTEGER TO NOTE MAPPING

int_to_note = dict((number, note) for number, note in enumerate(pitchnames))

print("Total Notes Mapping:", len(int_to_note))

Total Notes Mapping: 274


In [29]:
for i in range(500):

    prediction_input = np.reshape(pattern, (1, len(pattern), 1))
    prediction_input = prediction_input / float(len(pitchnames))

    prediction = model.predict(prediction_input, verbose=0)

    index = np.argmax(prediction)

    result = int_to_note[index]

    prediction_output.append(result)

    pattern = np.append(pattern, index)
    pattern = pattern[1:]

In [31]:
# CHECK GENERATED NOTES

In [32]:
print(len(prediction_output))

500


In [33]:
# SAVE GENERATED MIDI FILE

In [34]:
from music21 import instrument, note, chord, stream

offset = 0
output_notes = []

for pattern in prediction_output:

    if "." in pattern or pattern.isdigit():

        notes_in_chord = pattern.split(".")

        notes = []

        for current_note in notes_in_chord:
            new_note = note.Note(int(current_note))
            new_note.storedInstrument = instrument.Piano()
            notes.append(new_note)

        new_chord = chord.Chord(notes)
        new_chord.offset = offset
        output_notes.append(new_chord)

    else:

        new_note = note.Note(pattern)
        new_note.offset = offset
        new_note.storedInstrument = instrument.Piano()
        output_notes.append(new_note)

    offset += 0.5

midi_stream = stream.Stream(output_notes)
midi_stream.write("midi", fp="generated_music.mid")

print("Music Generated Successfully!")

Music Generated Successfully!


In [35]:
# PLAY GENERATED MUSIC

In [36]:
from IPython.display import Audio

Audio("generated_music.mid")

In [37]:
# VERIFY GENERATED FILE

In [38]:
import os

if os.path.exists("generated_music.mid"):
    print("✅ Music file generated successfully!")
else:
    print("❌ Music file not found.")

✅ Music file generated successfully!
